# Mean iLASH segment length vs GRM (and PC-Relate) kinship
One dot per pair with segments: notebook only, nothing saved.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
import glob
OUT = single_target(cfg, "ilash")
src = OUT if get_store(OUT).exists(OUT) else sorted(glob.glob(OUT.replace("pairs.tsv", "pairs.chr*.tsv")))[-1]   # full run, else the test run
pairs = pd.read_csv(src, sep="\t", dtype={"id1": str, "id2": str})
pairs["mean_cm"] = pairs["total_cm"] / pairs["n_segments"]
print(os.path.basename(src), len(pairs), "pairs with segments")

ids = pd.read_csv(target_items(cfg, "grm_shards")[0][0].split(".grm.bin.")[0] + ".grm.id", sep="\t", header=None, dtype=str)[1]
pos = pd.Series(np.arange(len(ids)), index=ids)
pairs = pairs[pairs["id1"].isin(pos.index) & pairs["id2"].isin(pos.index)]
a, b = pos[pairs["id1"]].to_numpy(), pos[pairs["id2"]].to_numpy()
t = np.maximum(a, b) * (np.maximum(a, b) + 1) // 2 + np.minimum(a, b)
grm = np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")])
kin = {"GRM / 2": grm[t] / 2}
pr = single_target(cfg, "pcrelate")
if get_store(pr).exists(pr):
    assert list(pd.read_csv(pr.replace(".bin", ".id"), header=None, dtype=str)[0]) == list(ids)
    kin["PC-Relate"] = np.fromfile(pr, dtype="float32")[t]
del grm

In [ ]:
fig, axes = plt.subplots(2, len(kin), figsize=(7 * len(kin), 10), squeeze=False)
for col, (name, k) in enumerate(kin.items()):
    for row, xlim in enumerate([(-0.03, 0.06), (-0.03, 0.55)]):
        ax = axes[row, col]
        sc = ax.scatter(k, pairs["mean_cm"], c=np.log10(pairs["n_segments"]), s=3, alpha=0.5, cmap="viridis", rasterized=True)
        ax.set(xlim=xlim, yscale="log", xlabel=f"{name} kinship", ylabel="mean iLASH segment length (cM)",
               title=f"{RUN_ID}: {name}" + (" (distant pairs)" if row == 0 else ""))
fig.colorbar(sc, ax=axes, label="log10 number of segments")
plt.show()